# Level-Specific Linear RRF Weights With Gemini LLM Judge

This notebook extends the global-weight experiment with **one weight vector per process level**:

1. Split UC1 and UC2 into training and validation queries per level.
2. Run BM25 + cross-encoder candidate ranking and RRF.
3. Let the Gemini LLM judge improve the four query variants.
4. Run BM25 + cross-encoder candidate ranking again and RRF.
5. Learn **separate** RRF weights for **subprocess** and **task** from UC1/UC2 training queries.
6. Evaluate **process** queries using the learned **subprocess** weights (process is not used for weight learning).
7. Apply the pipeline to validation (UC1/UC2) and external test (UC3).
8. Evaluate recall, precision, and accuracy.

Weights are pooled across UC1 and UC2 within each learnable level. Process is excluded from learning and reuses subprocess weights at evaluation time.

## Method

For each original query $q$, the pipeline uses five retrieval inputs:

$$
V = \{v_0, v_1, v_2, v_3, v_4\}
$$

For each variant $v_i$, BM25 retrieves candidates and the cross-encoder reranks with the original query $q$. The rank-based feature is:

$$
x_i(q,d) =
\begin{cases}
\frac{1}{K + r_i(q,d)} & \text{if document } d \text{ is retrieved by variant } v_i \\
0 & \text{otherwise}
\end{cases}
$$

Separate logistic regressions are trained for subprocess and task over all UC1/UC2 **training** queries at that level:

$$
P(y=1 \mid q,d, \ell) = \sigma\left(\beta_{0,\ell} + \boldsymbol{\beta}_\ell^\top \mathbf{x}(q,d)\right), \quad \ell \in \{\text{subprocess}, \text{task}\}
$$

Weights are clipped and normalized per level:

$$
w_{i,\ell} = \frac{\max(\beta_{i,\ell}, 0)}{\sum_j \max(\beta_{j,\ell}, 0)}
$$

Process-level evaluation reuses subprocess weights because process has only one query per use case and is excluded from learning:

$$
\mathbf{w}_{process} = \mathbf{w}_{subprocess}
$$

In [12]:
from pathlib import Path
import importlib
import os
import sys

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

project_root = Path.cwd()
while project_root.name != "Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval" and project_root.parent != project_root:
    project_root = project_root.parent

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import main
importlib.reload(main)

from main import JudgeContext, QueryVariants, clean_text, llm_judge_refine_queries
from retrieval.retrieval_bm25 import Query, build_bm25_index, load_corpus
from retrieval.sota_retrieval import SotaRetriever

print(f"Working directory: {Path.cwd()}")

Working directory: /Users/mareklorenz/Development/Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval


## Configuration

`RUN_LLM_JUDGE` controls whether missing judged variants are generated with Gemini. Judged variants are cached in `output_with_agents_<use_case>.csv` under `global_judge_*` columns.

`WEIGHT_LEARNING_LEVELS` defines where weights are learned. Process queries are evaluated with subprocess weights only.

In [13]:
TRAIN_USE_CASES = ["uc1", "uc2"]
EXTERNAL_TEST_USE_CASES = ["uc3"]
EVALUATION_LEVELS = ["process", "subprocess", "task"]
WEIGHT_LEARNING_LEVELS = ["subprocess", "task"]
PROCESS_WEIGHT_SOURCE_LEVEL = "subprocess"

RRF_K = 60
RANDOM_STATE = 42
JUDGE_PROVIDER = "gemini"
RUN_LLM_JUDGE = True

BASE_VARIANTS = [
    ("baseline", "query"),
    ("legal_terminology_rewrite", "legal_terminology_rewrite"),
    ("regulatory_compliance_query", "regulatory_compliance_query"),
    ("contract_clause_query", "contract_clause_query"),
    ("risk_scenario_query", "risk_scenario_query"),
]

GLOBAL_JUDGE_VARIANTS = [
    ("baseline", "query"),
    ("global_judge_legal_terminology_rewrite", "global_judge_legal_terminology_rewrite"),
    ("global_judge_regulatory_compliance_query", "global_judge_regulatory_compliance_query"),
    ("global_judge_contract_clause_query", "global_judge_contract_clause_query"),
    ("global_judge_risk_scenario_query", "global_judge_risk_scenario_query"),
]

LEVEL_TO_GS_SUFFIX = {
    "process": "process_level",
    "subprocess": "subprocess_level",
    "task": "event_level",
}

LEVEL_TO_TOP_K = {
    "process": 100,
    "subprocess": 30,
    "task": 15,
}

INITIAL_RRF_WEIGHTS = np.ones(len(BASE_VARIANTS)) / len(BASE_VARIANTS)

In [ ]:
def corpus_path_for(use_case):
    return Path(f"regulatory_relevance4process/SOTA_NLP_LIR/input_ranking/{use_case}/Input_corpus_{use_case}.xlsx")


def gold_path_for(use_case, level):
    suffix = LEVEL_TO_GS_SUFFIX[level]
    return Path(f"regulatory_relevance4process/SOTA_NLP_LIR/output_ranking_input_eval/{use_case}/gold_standard/gs_{use_case}_{suffix}.xlsx")


def recorded_path_for(use_case):
    return Path(f"output_with_agents_{use_case}.csv")


def load_records(use_case):
    path = recorded_path_for(use_case)
    if not path.exists():
        raise FileNotFoundError(f"Missing recorded query variants: {path}")
    df = pd.read_csv(path)
    df["level"] = df["level"].astype(str).str.lower()
    df["query_clean"] = df["query"].apply(clean_text)
    return df


def save_records(use_case, records_df):
    path = recorded_path_for(use_case)
    records_df.drop(columns=["query_clean"], errors="ignore").to_csv(path, index=False)
    print(f"Saved {path}")


def variant_text(row, column):
    if column not in row.index:
        return clean_text(row["query"])
    value = row[column]
    if value is None or (isinstance(value, float) and pd.isna(value)) or str(value).strip() == "":
        return clean_text(row["query"])
    return clean_text(value)


def split_queries(queries):
    queries = list(queries)
    if len(queries) == 1:
        return queries, queries, "single_query_reused"
    rng = np.random.default_rng(RANDOM_STATE)
    indices = np.arange(len(queries))
    rng.shuffle(indices)
    split_at = max(1, len(indices) // 2)
    train_indices = set(indices[:split_at])
    train_queries = [query for idx, query in enumerate(queries) if idx in train_indices]
    validation_queries = [query for idx, query in enumerate(queries) if idx not in train_indices]
    return train_queries, validation_queries, "random_half_split"


def weights_for_level(level, weights_by_level):
    if level in weights_by_level:
        return weights_by_level[level]
    if level == "process":
        return weights_by_level[PROCESS_WEIGHT_SOURCE_LEVEL]
    raise KeyError(f"No weights configured for level '{level}'")

## BM25 + CE + RRF Helpers

Each variant retrieves BM25 candidates. The cross-encoder reranks those candidates with the original query. RRF then combines the variant lists.

In [15]:
def ce_rank_variant_lists(row, variant_columns, bm25_index, documents, cross_encoder, top_k):
    original_query = clean_text(row["query"])
    ranked_lists = {}
    for variant_name, column in variant_columns:
        bm25_results = bm25_index.rank(Query(text=variant_text(row, column)), top_k=top_k)
        if not bm25_results:
            ranked_lists[variant_name] = []
            continue
        cross_inputs = [[original_query, documents[result.document.doc_id].text] for result in bm25_results]
        cross_scores = cross_encoder.predict(cross_inputs)
        scored = [
            (result.document.doc_id, float(score))
            for result, score in zip(bm25_results, cross_scores, strict=True)
        ]
        scored.sort(key=lambda item: item[1], reverse=True)
        ranked_lists[variant_name] = [
            {"doc_id": doc_id, "score": score, "rank": rank}
            for rank, (doc_id, score) in enumerate(scored, start=1)
        ]
    return ranked_lists


def rrf_feature_rows(ranked_lists, variant_columns):
    doc_features = {}
    for variant_index, (variant_name, _) in enumerate(variant_columns):
        for item in ranked_lists.get(variant_name, []):
            features = doc_features.setdefault(item["doc_id"], np.zeros(len(variant_columns), dtype=float))
            features[variant_index] = 1.0 / (RRF_K + item["rank"])
    return doc_features


def rank_with_rrf(row, variant_columns, weights, bm25_index, documents, cross_encoder, top_k):
    ranked_lists = ce_rank_variant_lists(row, variant_columns, bm25_index, documents, cross_encoder, top_k)
    doc_features = rrf_feature_rows(ranked_lists, variant_columns)
    scored_docs = [(doc_id, float(np.dot(features, weights))) for doc_id, features in doc_features.items()]
    scored_docs.sort(key=lambda item: item[1], reverse=True)
    return scored_docs[:top_k], ranked_lists, doc_features


def ranked_docs_to_context_df(scored_docs, documents, query, level):
    return pd.DataFrame(
        [
            {
                "level": level,
                "query": query,
                "rel_text": clean_text(documents[doc_id].text),
                "score": score,
                "method": "rrf_ce_context",
                "query_variant": "rrf_ce_context",
                "source_variants": "rrf_ce_context",
            }
            for doc_id, score in scored_docs
        ]
    )

## Gemini LLM Judge Query Improvement

The judge receives the top 15 original BM25 results and top 15 initial RRF+CE results, then rewrites the four non-baseline variants.

In [16]:
def base_variants_from_row(row):
    return QueryVariants(
        legal_terminology_rewrite=variant_text(row, "legal_terminology_rewrite"),
        regulatory_compliance_query=variant_text(row, "regulatory_compliance_query"),
        contract_clause_query=variant_text(row, "contract_clause_query"),
        risk_scenario_query=variant_text(row, "risk_scenario_query"),
    )


def judge_columns_missing(row):
    required = [column for _, column in GLOBAL_JUDGE_VARIANTS if column != "query"]
    return any(column not in row.index or pd.isna(row[column]) or str(row[column]).strip() == "" for column in required)


def improve_row_queries(row, level, bm25_index, documents, sota_retriever, top_k):
    if not RUN_LLM_JUDGE:
        raise ValueError("Missing global judge variants. Set RUN_LLM_JUDGE = True to generate them.")

    initial_ranked, _, _ = rank_with_rrf(
        row=row,
        variant_columns=BASE_VARIANTS,
        weights=INITIAL_RRF_WEIGHTS,
        bm25_index=bm25_index,
        documents=documents,
        cross_encoder=sota_retriever.cross_encoder,
        top_k=top_k,
    )
    baseline_bm25 = bm25_index.rank(Query(text=clean_text(row["query"])), top_k=top_k)
    ce_context = ranked_docs_to_context_df(initial_ranked[:15], documents, clean_text(row["query"]), level)
    return llm_judge_refine_queries(
        JudgeContext(
            original_query=clean_text(row["query"]),
            variants=base_variants_from_row(row),
            bm25_results=baseline_bm25[:15],
            ce_rows=ce_context,
        ),
        documents=documents,
        judge_provider=JUDGE_PROVIDER,
    )


def ensure_global_judge_variants(use_case, records_df):
    required = [column for _, column in GLOBAL_JUDGE_VARIANTS if column != "query"]
    for column in required:
        if column not in records_df.columns:
            records_df[column] = ""

    missing_mask = records_df.apply(judge_columns_missing, axis=1)
    if not missing_mask.any():
        print(f"{use_case}: global judge variants already available.")
        return records_df

    documents = load_corpus(str(corpus_path_for(use_case)))
    bm25_index = build_bm25_index(str(corpus_path_for(use_case)))
    sota_retriever = SotaRetriever([document.text for document in documents])

    for row_index, row in records_df[missing_mask].iterrows():
        level = row["level"]
        top_k = LEVEL_TO_TOP_K[level]
        print(f"[{use_case} / {level}] Gemini judge query improvement for row {row_index}")
        judged = improve_row_queries(row, level, bm25_index, documents, sota_retriever, top_k)
        records_df.loc[row_index, "global_judge_legal_terminology_rewrite"] = judged.legal_terminology_rewrite
        records_df.loc[row_index, "global_judge_regulatory_compliance_query"] = judged.regulatory_compliance_query
        records_df.loc[row_index, "global_judge_contract_clause_query"] = judged.contract_clause_query
        records_df.loc[row_index, "global_judge_risk_scenario_query"] = judged.risk_scenario_query

    save_records(use_case, records_df)
    return records_df

## Split UC1/UC2 And Learn Level-Specific Weight Vectors

Training examples are built only from **subprocess** and **task** training splits. Process splits are recorded for evaluation but excluded from logistic regression.

In [17]:
def build_gold_lookup(use_case, level, documents):
    gold_df = pd.read_excel(gold_path_for(use_case, level))
    gold_df["query_clean"] = gold_df["query"].apply(clean_text)
    doc_id_by_text = {clean_text(document.text): document.doc_id for document in documents}

    gold_lookup = {}
    for query, group in gold_df.groupby("query_clean"):
        gold_doc_ids = {
            doc_id_by_text[clean_text(rel_text)]
            for rel_text in group["rel_text"].tolist()
            if clean_text(rel_text) in doc_id_by_text
        }
        gold_lookup[query] = gold_doc_ids
    return gold_lookup


def level_queries(records_df, gold_lookup, level):
    level_records = records_df[records_df["level"] == level]
    return [query for query in level_records["query_clean"].tolist() if query in gold_lookup]


def build_examples_for_queries(use_case, level, records_df, query_subset):
    top_k = LEVEL_TO_TOP_K[level]
    documents = load_corpus(str(corpus_path_for(use_case)))
    bm25_index = build_bm25_index(str(corpus_path_for(use_case)))
    sota_retriever = SotaRetriever([document.text for document in documents])
    gold_lookup = build_gold_lookup(use_case, level, documents)
    level_records = records_df[records_df["level"] == level].copy()

    examples = []
    labels = []
    for query in query_subset:
        row = level_records[level_records["query_clean"] == query].iloc[0]
        _, _, doc_features = rank_with_rrf(
            row=row,
            variant_columns=GLOBAL_JUDGE_VARIANTS,
            weights=INITIAL_RRF_WEIGHTS,
            bm25_index=bm25_index,
            documents=documents,
            cross_encoder=sota_retriever.cross_encoder,
            top_k=top_k,
        )
        positives = gold_lookup.get(query, set())
        for doc_id, features in doc_features.items():
            examples.append(features)
            labels.append(1 if doc_id in positives else 0)
    return examples, labels


records_by_use_case = {}
splits = []
examples_by_level = {level: [] for level in WEIGHT_LEARNING_LEVELS}
labels_by_level = {level: [] for level in WEIGHT_LEARNING_LEVELS}

for use_case in TRAIN_USE_CASES:
    records_df = ensure_global_judge_variants(use_case, load_records(use_case))
    records_by_use_case[use_case] = records_df
    documents = load_corpus(str(corpus_path_for(use_case)))
    for level in EVALUATION_LEVELS:
        gold_lookup = build_gold_lookup(use_case, level, documents)
        queries = level_queries(records_df, gold_lookup, level)
        train_queries, validation_queries, split_note = split_queries(queries)
        splits.append(
            {
                "use_case": use_case,
                "level": level,
                "split_note": split_note,
                "used_for_weight_learning": level in WEIGHT_LEARNING_LEVELS,
                "train_queries": train_queries,
                "validation_queries": validation_queries,
            }
        )
        if level in WEIGHT_LEARNING_LEVELS:
            examples, labels = build_examples_for_queries(use_case, level, records_df, train_queries)
            examples_by_level[level].extend(examples)
            labels_by_level[level].extend(labels)

for level in WEIGHT_LEARNING_LEVELS:
    print(f"{level}: training examples={len(examples_by_level[level])}, positive labels={int(np.sum(labels_by_level[level]))}")

uc1: global judge variants already available.
uc2: global judge variants already available.
subprocess: training examples=453, positive labels=30
task: training examples=973, positive labels=40


In [18]:
def learn_level_weights(x_train, y_train):
    if len(x_train) == 0 or len(np.unique(y_train)) < 2:
        return np.ones(len(GLOBAL_JUDGE_VARIANTS)) / len(GLOBAL_JUDGE_VARIANTS), "equal_weights_fallback"

    model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)
    model.fit(x_train, y_train)
    raw_weights = np.maximum(model.coef_[0], 0.0)
    if raw_weights.sum() == 0:
        return np.ones(len(GLOBAL_JUDGE_VARIANTS)) / len(GLOBAL_JUDGE_VARIANTS), "equal_weights_fallback"
    return raw_weights / raw_weights.sum(), "level_linear_logistic_coefficients"


weights_by_level = {}
weight_sources_by_level = {}
weight_rows = []

for level in WEIGHT_LEARNING_LEVELS:
    x_train = np.array(examples_by_level[level])
    y_train = np.array(labels_by_level[level])
    weights, weight_source = learn_level_weights(x_train, y_train)
    weights_by_level[level] = weights
    weight_sources_by_level[level] = weight_source
    weight_rows.append(
        {
            "level": level,
            "weight_source": weight_source,
            "learned_from_level": level,
            "training_use_cases": ",".join(TRAIN_USE_CASES),
            "training_examples": len(x_train),
            "positive_labels": int(y_train.sum()),
            **{variant_name: weight for (variant_name, _), weight in zip(GLOBAL_JUDGE_VARIANTS, weights)},
        }
    )

weights_by_level["process"] = weights_by_level[PROCESS_WEIGHT_SOURCE_LEVEL].copy()
weight_sources_by_level["process"] = f"reused_{PROCESS_WEIGHT_SOURCE_LEVEL}_weights"
weight_rows.append(
    {
        "level": "process",
        "weight_source": weight_sources_by_level["process"],
        "learned_from_level": PROCESS_WEIGHT_SOURCE_LEVEL,
        "training_use_cases": ",".join(TRAIN_USE_CASES),
        "training_examples": 0,
        "positive_labels": 0,
        **{
            variant_name: weight
            for (variant_name, _), weight in zip(GLOBAL_JUDGE_VARIANTS, weights_by_level["process"])
        },
    }
)

weights_df = pd.DataFrame(weight_rows)
weights_df

,level,weight_source,learned_from_level,training_use_cases,training_examples,positive_labels,baseline,global_judge_legal_terminology_rewrite,global_judge_regulatory_compliance_query,global_judge_contract_clause_query,global_judge_risk_scenario_query
0,subprocess,level_linear_logistic_coefficients,subprocess,"uc1,uc2",453,30,0.085944,0.176928,0.337173,0.170370,0.229585
1,task,level_linear_logistic_coefficients,task,"uc1,uc2",973,40,0.178864,0.235117,0.240922,0.164092,0.181005
2,process,reused_subprocess_weights,subprocess,"uc1,uc2",0,0,0.085944,0.176928,0.337173,0.170370,0.229585


## Verify Level Weights vs Global Baseline

This cell checks that subprocess/task weights are **not** identical to the single global weight vector. Similar final metrics can still occur when the learned weights change scores but not the top-k document sets.

In [19]:
GLOBAL_WEIGHTS_REFERENCE = np.array([0.178309, 0.223762, 0.246037, 0.168128, 0.183764])
GLOBAL_RESULTS_PATH = project_root / "linear_rrf_global_weights_llm_judge_results.xlsx"

if GLOBAL_RESULTS_PATH.exists():
    global_weights_df = pd.read_excel(GLOBAL_RESULTS_PATH, sheet_name="global_weights")
    global_weights_reference = global_weights_df[
        [name for name, _ in GLOBAL_JUDGE_VARIANTS]
    ].iloc[0].to_numpy(dtype=float)
else:
    global_weights_reference = GLOBAL_WEIGHTS_REFERENCE

comparison_rows = []
for level in ["subprocess", "task", "process"]:
    level_weights = weights_for_level(level, weights_by_level)
    l1_distance = float(np.abs(level_weights - global_weights_reference).sum())
    comparison_rows.append(
        {
            "level": level,
            "learned_from_level": level if level in WEIGHT_LEARNING_LEVELS else PROCESS_WEIGHT_SOURCE_LEVEL,
            "l1_distance_to_global": l1_distance,
            "max_component_delta": float(np.abs(level_weights - global_weights_reference).max()),
            "uses_same_vector_as_global": bool(np.allclose(level_weights, global_weights_reference)),
        }
    )

weights_comparison_df = pd.DataFrame(comparison_rows)
print("Weight vectors differ from global (expected for subprocess; task is often close):")
weights_comparison_df

Weight vectors differ from global (expected for subprocess; task is often close):


,level,learned_from_level,l1_distance_to_global,max_component_delta,uses_same_vector_as_global
0,subprocess,subprocess,0.278398,0.092365,False
1,task,task,0.023820,0.011355,False
2,process,subprocess,0.278398,0.092365,False


## Apply Pipeline To Test Data

Test data consists of:

- UC1 validation split (all levels)
- UC2 validation split (all levels)
- all UC3 queries (external test)

Each level uses its own weight vector. Process queries use subprocess weights.

In [20]:
def evaluate_prediction_sets(predictions, gold_lookup, corpus_size):
    tp = fp = fn = tn = 0
    for query, predicted in predictions.items():
        gold = gold_lookup.get(query, set())
        tp += len(predicted & gold)
        fp += len(predicted - gold)
        fn += len(gold - predicted)
        tn += corpus_size - len(gold | predicted)

    accuracy = (tp + tn) / (tp + fp + fn + tn) if tp + fp + fn + tn else np.nan
    precision = tp / (tp + fp) if tp + fp else np.nan
    recall = tp / (tp + fn) if tp + fn else np.nan
    return {
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
    }


def apply_level_pipeline(use_case, level, records_df, query_subset, split_name, level_weights):
    top_k = LEVEL_TO_TOP_K[level]
    documents = load_corpus(str(corpus_path_for(use_case)))
    bm25_index = build_bm25_index(str(corpus_path_for(use_case)))
    sota_retriever = SotaRetriever([document.text for document in documents])
    gold_lookup = build_gold_lookup(use_case, level, documents)
    level_records = records_df[records_df["level"] == level].copy()
    weight_source = weight_sources_by_level.get(level, f"reused_{PROCESS_WEIGHT_SOURCE_LEVEL}_weights")
    learned_from_level = level if level in WEIGHT_LEARNING_LEVELS else PROCESS_WEIGHT_SOURCE_LEVEL

    predictions = {}
    rows = []
    for query in query_subset:
        row = level_records[level_records["query_clean"] == query].iloc[0]
        ranked_docs, _, _ = rank_with_rrf(
            row=row,
            variant_columns=GLOBAL_JUDGE_VARIANTS,
            weights=level_weights,
            bm25_index=bm25_index,
            documents=documents,
            cross_encoder=sota_retriever.cross_encoder,
            top_k=top_k,
        )
        predictions[query] = {doc_id for doc_id, _ in ranked_docs}
        for rank, (doc_id, score) in enumerate(ranked_docs, start=1):
            rows.append(
                {
                    "use_case": use_case,
                    "level": level,
                    "learned_from_level": learned_from_level,
                    "weight_source": weight_source,
                    "split": split_name,
                    "query": query,
                    "rank": rank,
                    "doc_id": doc_id,
                    "score": score,
                    "rel_text": clean_text(documents[doc_id].text),
                }
            )

    metrics = evaluate_prediction_sets(predictions, gold_lookup, len(documents))
    metrics.update(
        {
            "use_case": use_case,
            "level": level,
            "learned_from_level": learned_from_level,
            "weight_source": weight_source,
            "split": split_name,
            "queries": len(query_subset),
        }
    )
    return metrics, rows


metric_rows = []
prediction_rows = []

for split in splits:
    level = split["level"]
    level_weights = weights_for_level(level, weights_by_level)
    metrics, rows = apply_level_pipeline(
        use_case=split["use_case"],
        level=level,
        records_df=records_by_use_case[split["use_case"]],
        query_subset=split["validation_queries"],
        split_name="validation",
        level_weights=level_weights,
    )
    metric_rows.append(metrics)
    prediction_rows.extend(rows)

for use_case in EXTERNAL_TEST_USE_CASES:
    records_df = ensure_global_judge_variants(use_case, load_records(use_case))
    records_by_use_case[use_case] = records_df
    documents = load_corpus(str(corpus_path_for(use_case)))
    for level in EVALUATION_LEVELS:
        gold_lookup = build_gold_lookup(use_case, level, documents)
        queries = level_queries(records_df, gold_lookup, level)
        level_weights = weights_for_level(level, weights_by_level)
        metrics, rows = apply_level_pipeline(
            use_case=use_case,
            level=level,
            records_df=records_df,
            query_subset=queries,
            split_name="external_test",
            level_weights=level_weights,
        )
        metric_rows.append(metrics)
        prediction_rows.extend(rows)

metrics_df = pd.DataFrame(metric_rows)
predictions_df = pd.DataFrame(prediction_rows)
metrics_df

uc3: global judge variants already available.


,true_positives,false_positives,false_negatives,true_negatives,accuracy,precision,recall,use_case,level,learned_from_level,weight_source,split,queries
0,15,85,34,355,0.756646,0.150000,0.306122,uc1,process,subprocess,reused_subprocess_weights,validation,1
1,10,110,28,1808,0.929448,0.083333,0.263158,uc1,subprocess,subprocess,level_linear_logistic_coefficients,validation,4
2,19,206,54,7056,0.964554,0.084444,0.260274,uc1,task,task,level_linear_logistic_coefficients,validation,15
3,26,74,5,206,0.745981,0.260000,0.838710,uc2,process,subprocess,reused_subprocess_weights,validation,1
4,22,98,18,1106,0.906752,0.183333,0.550000,uc2,subprocess,subprocess,level_linear_logistic_coefficients,validation,4
5,27,123,38,2922,0.948232,0.180000,0.415385,uc2,task,task,level_linear_logistic_coefficients,validation,10
6,17,83,0,68,0.505952,0.170000,1.000000,uc3,process,subprocess,reused_subprocess_weights,external_test,1
7,32,118,15,675,0.841667,0.213333,0.680851,uc3,subprocess,subprocess,level_linear_logistic_coefficients,external_test,5
8,44,256,39,3021,0.912202,0.146667,0.530120,uc3,task,task,level_linear_logistic_coefficients,external_test,20


## Ranking Sensitivity Check

For validation queries, compare top-k predictions under level-specific weights vs the global weight vector. This explains why metrics can look almost identical even when weights differ.

In [21]:
def predictions_for_queries(use_case, level, records_df, query_subset, weights):
    top_k = LEVEL_TO_TOP_K[level]
    documents = load_corpus(str(corpus_path_for(use_case)))
    bm25_index = build_bm25_index(str(corpus_path_for(use_case)))
    sota_retriever = SotaRetriever([document.text for document in documents])
    level_records = records_df[records_df["level\"] == level].copy()

    predictions = {}
    for query in query_subset:
        row = level_records[level_records["query_clean\"] == query].iloc[0]
        ranked_docs, _, _ = rank_with_rrf(
            row=row,
            variant_columns=GLOBAL_JUDGE_VARIANTS,
            weights=weights,
            bm25_index=bm25_index,
            documents=documents,
            cross_encoder=sota_retriever.cross_encoder,
            top_k=top_k,
        )
        predictions[query] = {doc_id for doc_id, _ in ranked_docs}
    return predictions


sensitivity_rows = []
for split in splits:
    if split["use_case"] not in TRAIN_USE_CASES:
        continue
    level = split["level"]
    validation_queries = split["validation_queries"]
    if not validation_queries:
        continue

    level_weights = weights_for_level(level, weights_by_level)
    level_predictions = predictions_for_queries(
        split["use_case"], level, records_by_use_case[split["use_case"]], validation_queries, level_weights
    )
    global_predictions = predictions_for_queries(
        split["use_case"], level, records_by_use_case[split["use_case"]], validation_queries, global_weights_reference
    )

    changed = sum(level_predictions[q] != global_predictions[q] for q in validation_queries)
    sensitivity_rows.append(
        {
            "use_case": split["use_case"],
            "level": level,
            "validation_queries": len(validation_queries),
            "queries_with_different_top_k": changed,
            "fraction_changed": changed / len(validation_queries),
            "applied_weight_level": level if level in WEIGHT_LEARNING_LEVELS else PROCESS_WEIGHT_SOURCE_LEVEL,
        }
    )

sensitivity_df = pd.DataFrame(sensitivity_rows)
sensitivity_df

SyntaxError: unterminated string literal (detected at line 6) (427815805.py, line 6)

## Summary Table

Recall, precision, and accuracy per use case and level. Process rows use subprocess weights.

In [ ]:
summary_df = metrics_df[
    [
        "use_case",
        "level",
        "learned_from_level",
        "weight_source",
        "split",
        "queries",
        "true_positives",
        "false_positives",
        "false_negatives",
        "true_negatives",
        "recall",
        "precision",
        "accuracy",
    ]
].copy()
summary_df[["recall", "precision", "accuracy"]] = summary_df[["recall", "precision", "accuracy"]].round(3)
summary_df

,use_case,level,learned_from_level,weight_source,split,queries,true_positives,false_positives,false_negatives,true_negatives,recall,precision,accuracy
0,uc1,process,subprocess,reused_subprocess_weights,validation,1,15,85,34,355,0.306,0.150,0.757
1,uc1,subprocess,subprocess,level_linear_logistic_coefficients,validation,4,10,110,28,1808,0.263,0.083,0.929
2,uc1,task,task,level_linear_logistic_coefficients,validation,15,19,206,54,7056,0.260,0.084,0.965
3,uc2,process,subprocess,reused_subprocess_weights,validation,1,26,74,5,206,0.839,0.260,0.746
4,uc2,subprocess,subprocess,level_linear_logistic_coefficients,validation,4,22,98,18,1106,0.550,0.183,0.907
5,uc2,task,task,level_linear_logistic_coefficients,validation,10,27,123,38,2922,0.415,0.180,0.948
6,uc3,process,subprocess,reused_subprocess_weights,external_test,1,17,83,0,68,1.000,0.170,0.506
7,uc3,subprocess,subprocess,level_linear_logistic_coefficients,external_test,5,32,118,15,675,0.681,0.213,0.842
8,uc3,task,task,level_linear_logistic_coefficients,external_test,20,44,256,39,3021,0.530,0.147,0.912


## Export Results

The workbook stores level-specific weight vectors, split definitions, evaluation metrics, and ranked predictions.

In [ ]:
export_path = project_root / "linear_rrf_level_weights_llm_judge_gemini_results.xlsx"

split_rows = []
for split in splits:
    split_rows.append(
        {
            "use_case": split["use_case"],
            "level": split["level"],
            "split_note": split["split_note"],
            "used_for_weight_learning": split["used_for_weight_learning"],
            "train_queries": len(split["train_queries"]),
            "validation_queries": len(split["validation_queries"]),
        }
    )
splits_df = pd.DataFrame(split_rows)

with pd.ExcelWriter(export_path) as writer:
    weights_df.to_excel(writer, sheet_name="level_weights", index=False)
    splits_df.to_excel(writer, sheet_name="splits", index=False)
    metrics_df.to_excel(writer, sheet_name="metrics", index=False)
    predictions_df.to_excel(writer, sheet_name="ranked_predictions", index=False)

print(f"Exported {export_path}")

Exported /Users/mareklorenz/Development/Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval/linear_rrf_level_weights_llm_judge_gemini_results.xlsx
